# 📘 Notebook 04: Data Warehouses

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module A: Data at Rest · Notebook 4 of 4 in this module · (4 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Distinguish **transactional** workloads from **analytical** ones, and say why they pull a design in opposite directions
- Describe a **star schema**: a fact table, its **grain**, its **measures** and its **dimensions**
- Build a small data warehouse from an operational database with an **ETL** process written by hand
- Answer analytical questions by **rolling up**, **drilling down** and **slicing**
- Measure what **denormalisation** gains and what it costs
- Explain the problem of **history**, and keep it with a slowly changing dimension

> **Prerequisites:** Notebooks 01, 02 and 03.
>
> 🔭 **Why this notebook matters:** the database of the last three notebooks was designed to **run** the bookshop: record an order, look up a customer, change an address. A different group of people wants to **understand** the bookshop: which categories are growing, which cities buy at weekends, how this spring compares with the last. Their questions read millions of rows at a time and change nothing. A system tuned for the first kind of work is poor at the second, so organisations keep a second copy of their data, arranged differently, for analysis. That copy is a data warehouse, and its design is one of the most durable ideas in the field.

> 📦 **Libraries used in this notebook.** None that need installing: only **SQLite**, built into Python and introduced in Notebook 01.

In [ ]:
import random
import datetime
import sqlite3
import time

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]
    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

def build_database(customer_rows, product_rows, order_rows, item_rows):
    db = sqlite3.connect(":memory:")
    db.executescript("""
        CREATE TABLE customers   (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT, birth_year INTEGER);
        CREATE TABLE products    (product_id INTEGER PRIMARY KEY, title TEXT NOT NULL, category TEXT, price REAL);
        CREATE TABLE orders      (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL, order_date TEXT);
        CREATE TABLE order_items (item_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL, product_id INTEGER NOT NULL, quantity INTEGER);
        CREATE INDEX orders_by_customer ON orders(customer_id);
        CREATE INDEX items_by_order ON order_items(order_id);
    """)
    db.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customer_rows)
    db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", product_rows)
    db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
    db.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", item_rows)
    db.commit()
    return db

customer_rows, product_rows, order_rows, item_rows = make_shop(customers=5000, orders=100000)
shop = build_database(customer_rows, product_rows, order_rows, item_rows)

def show(db, sql, limit=8):
    cursor = db.execute(sql)
    columns = [description[0] for description in cursor.description]
    rows = cursor.fetchall()
    print(" | ".join(f"{column:<13}" for column in columns))
    print("-" * (16 * len(columns) - 3))
    for row in rows[:limit]:
        print(" | ".join(f"{str(value):<13}" for value in row))
    if len(rows) > limit:
        print(f"... {len(rows):,} rows in total")
    print()

print(f"{len(customer_rows):,} customers, {len(order_rows):,} orders, {len(item_rows):,} order lines")

## 1. Two kinds of work

### Intuition first
Picture two people using the bookshop's data at the same moment.

At the till, an assistant sells a book. The system must find one customer, add one order with two lines, and do it in a few milliseconds, correctly, while a hundred other tills do the same.

Upstairs, an analyst asks: *"What was our revenue from each category in each quarter, and how do weekends compare with weekdays?"* The system must read every order line of the last two years. Nobody minds if it takes a few seconds. Nothing is changed.

| | Running the business | Understanding the business |
|---|---|---|
| Name | **OLTP**: online transaction processing | **OLAP**: online analytical processing |
| A typical request | record one order, find one customer | total sales by category and quarter |
| Rows touched | a handful | millions |
| Reads or writes | many small writes | reads only |
| What matters | each request is fast and never corrupts the data | large summaries finish in reasonable time |
| Best design | **normalised**: every fact once (Notebook 01) | arranged for reading: this notebook |

The normalised design of Notebook 01 is ideal for the left-hand column. Let us see how it handles a question from the right-hand one.

In [ ]:
def timed(db, sql, repeats=3):
    started = time.perf_counter()
    for i in range(repeats):
        rows = db.execute(sql).fetchall()
    return rows, (time.perf_counter() - started) / repeats * 1000

operational_query = """
    SELECT p.category, SUBSTR(o.order_date, 1, 4) AS year, ROUND(SUM(i.quantity * p.price)) AS revenue
    FROM order_items AS i
    JOIN orders AS o   ON o.order_id = i.order_id
    JOIN products AS p ON p.product_id = i.product_id
    GROUP BY p.category, year
    ORDER BY p.category, year
"""
operational_rows, operational_time = timed(shop, operational_query)
show(shop, operational_query, limit=4)
print(f"Time on the operational database: {operational_time:.0f} ms")

It works. Notice what the analyst had to know, though: that revenue is not stored anywhere and must be computed from a quantity in one table and a price in another, that the date lives in a third table, and that the year has to be cut out of a text column. Every analytical question repeats that work, and every analyst must get it right.

There is a subtler problem too. Suppose the price of a book is raised next month. This query will then compute **last year's** revenue with the **new** price, and the past will quietly change. An operational database stores how things are **now**. An analyst needs to know how they **were**.

## 2. The star schema

### Intuition first
Every analytical question has the same shape: *some number, broken down by some descriptions.* Revenue **by** category. Copies sold **by** city and month. Average order value **by** age group.

So arrange the data to match the question.

- Put the **numbers** in one long, narrow table, one row for each event that was measured. This is the **fact table**.
- Put the **descriptions** in a few small, wide tables around it: one for dates, one for customers, one for products. These are the **dimension tables**.

Drawn on paper, the fact table sits in the middle with the dimensions around it like the points of a star.

```
                      dim_date
                  (date_key, date, year, quarter,
                   month, month_name, weekday, is_weekend)
                          │
                          │
 dim_customer ────── fact_sales ────── dim_product
 (customer_key,     (date_key,        (product_key,
  name, city,        customer_key,     title, category,
  age_group)         product_key,      price_band)
                     quantity,
                     revenue)
```

### Formal definition
- A **fact table** records events. Its **grain** is the precise meaning of one row. Ours is: *one line of one order*.
- Its **measures** are the numeric columns that get added up: `quantity` and `revenue`.
- Its remaining columns are **foreign keys** to the dimensions.
- A **dimension table** holds the descriptive attributes by which the measures are grouped and filtered. Dimensions are deliberately **denormalised**: facts that could be derived, such as the quarter of a date, are stored ready for use.

Three decisions here differ sharply from Notebook 01. Revenue is **stored**, at the price that applied on the day. The date gets a table of its own, with every way of describing a day worked out in advance. And attributes are added purely because analysts group by them, such as an age group and a price band.

In [ ]:
warehouse = sqlite3.connect(":memory:")
warehouse.executescript("""
    CREATE TABLE dim_date     (date_key INTEGER PRIMARY KEY, date TEXT, year INTEGER, quarter TEXT, month INTEGER,
                               month_name TEXT, weekday TEXT, is_weekend INTEGER);
    CREATE TABLE dim_customer (customer_key INTEGER PRIMARY KEY, name TEXT, city TEXT, age_group TEXT);
    CREATE TABLE dim_product  (product_key INTEGER PRIMARY KEY, title TEXT, category TEXT, price_band TEXT);
    CREATE TABLE fact_sales   (date_key INTEGER, customer_key INTEGER, product_key INTEGER, quantity INTEGER, revenue REAL);
""")
print("Warehouse tables created.")

## 3. Filling the warehouse: ETL

Data reaches a warehouse through a process with three steps, known by its initials.

- **Extract**: read the data from the operational system.
- **Transform**: clean it, derive new attributes, and reshape it into facts and dimensions.
- **Load**: write it into the warehouse.

### The date dimension
This table is not extracted from anywhere. It is **generated**: one row for every day, with everything an analyst might want to know about that day.

In [ ]:
def date_rows(first_day, last_day):
    rows = []
    day = first_day
    while day <= last_day:
        rows.append((
            int(day.strftime("%Y%m%d")),                 # date_key, for example 20240315
            day.isoformat(),
            day.year,
            f"{day.year}-Q{(day.month - 1) // 3 + 1}",
            day.month,
            day.strftime("%B"),
            day.strftime("%A"),
            1 if day.weekday() >= 5 else 0,              # Saturday and Sunday
        ))
        day += datetime.timedelta(days=1)
    return rows

warehouse.executemany("INSERT INTO dim_date VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                      date_rows(datetime.date(2023, 1, 1), datetime.date(2024, 12, 31)))
show(warehouse, "SELECT * FROM dim_date WHERE date BETWEEN '2024-03-14' AND '2024-03-17'")

The questions *"is this a weekend?"* and *"which quarter is this?"* have been answered once, for every day, by someone who got them right. No analyst will ever have to cut a year out of a text column again.

### The customer and product dimensions
Here the transform step derives the attributes used for grouping.

In [ ]:
def age_group(birth_year, today=2024):
    age = today - birth_year
    if age < 30:
        return "under 30"
    if age < 50:
        return "30 to 49"
    return "50 and over"

def price_band(price):
    if price < 20:
        return "under 20"
    if price < 40:
        return "20 to 39.99"
    return "40 and over"

extracted_customers = shop.execute("SELECT customer_id, name, city, birth_year FROM customers").fetchall()      # extract
transformed_customers = [(key, name, city, age_group(birth_year)) for key, name, city, birth_year in extracted_customers]   # transform
warehouse.executemany("INSERT INTO dim_customer VALUES (?, ?, ?, ?)", transformed_customers)                   # load

extracted_products = shop.execute("SELECT product_id, title, category, price FROM products").fetchall()
warehouse.executemany("INSERT INTO dim_product VALUES (?, ?, ?, ?)",
                      [(key, title, category, price_band(price)) for key, title, category, price in extracted_products])

show(warehouse, "SELECT * FROM dim_customer", limit=3)
show(warehouse, "SELECT * FROM dim_product", limit=3)

### The fact table
One row per order line. The revenue is computed **now**, from the price in force, and stored. Whatever happens to prices later, this row will not change.

In [ ]:
extracted_sales = shop.execute("""
    SELECT o.order_date, o.customer_id, i.product_id, i.quantity, p.price
    FROM order_items AS i
    JOIN orders AS o   ON o.order_id = i.order_id
    JOIN products AS p ON p.product_id = i.product_id
""").fetchall()

fact_rows = [(int(order_date.replace("-", "")), customer_id, product_id, quantity, round(quantity * price, 2))
             for order_date, customer_id, product_id, quantity, price in extracted_sales]

warehouse.executemany("INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?)", fact_rows)
warehouse.commit()

for table in ["fact_sales", "dim_date", "dim_customer", "dim_product"]:
    print(f"{table:<13} {warehouse.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:>8,} rows")

The shape is typical: one very long table of facts, and a few short tables of descriptions.

## 4. Asking analytical questions

Every query against a star schema has the same pattern: start from the fact table, join the dimensions you need, group by their attributes, and add up a measure.

The question of Section 1 again.

In [ ]:
star_query = """
    SELECT p.category, d.year, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f
    JOIN dim_product AS p ON p.product_key = f.product_key
    JOIN dim_date AS d    ON d.date_key = f.date_key
    GROUP BY p.category, d.year
    ORDER BY p.category, d.year
"""
star_rows, star_time = timed(warehouse, star_query)
show(warehouse, star_query, limit=4)

print("Same answer as the operational database:", [tuple(row) for row in star_rows] == [(c, int(y), r) for c, y, r in operational_rows])

The same numbers, from a query that reads like the question.

Analysts move around the data with a small set of standard moves.

**Roll up**: summarise at a coarser level. From categories and years, up to years alone.

In [ ]:
show(warehouse, """
    SELECT d.year, ROUND(SUM(f.revenue)) AS revenue, SUM(f.quantity) AS copies
    FROM fact_sales AS f JOIN dim_date AS d ON d.date_key = f.date_key
    GROUP BY d.year
""")

**Drill down**: go to a finer level. From years, down to quarters.

In [ ]:
show(warehouse, """
    SELECT d.quarter, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f JOIN dim_date AS d ON d.date_key = f.date_key
    GROUP BY d.quarter
    ORDER BY d.quarter
""", limit=8)

**Slice**: fix one value of a dimension, and look at the rest. Only Heraklion, by category.

In [ ]:
show(warehouse, """
    SELECT p.category, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f
    JOIN dim_product AS p  ON p.product_key = f.product_key
    JOIN dim_customer AS c ON c.customer_key = f.customer_key
    WHERE c.city = 'Heraklion'
    GROUP BY p.category
    ORDER BY revenue DESC
""", limit=4)

And a question that would have been awkward in the operational database becomes plain, because the date dimension did the thinking in advance: how do weekends compare with weekdays, for each age group?

In [ ]:
show(warehouse, """
    SELECT c.age_group,
           CASE d.is_weekend WHEN 1 THEN 'weekend' ELSE 'weekday' END AS part_of_week,
           COUNT(*) AS order_lines,
           ROUND(AVG(f.revenue), 2) AS average_line_value
    FROM fact_sales AS f
    JOIN dim_customer AS c ON c.customer_key = f.customer_key
    JOIN dim_date AS d     ON d.date_key = f.date_key
    GROUP BY c.age_group, part_of_week
    ORDER BY c.age_group, part_of_week
""")

(Our invented data has no real pattern here, so the averages are nearly equal. With real sales they rarely are.)

## 5. What does the rearrangement gain, and cost?

A warehouse stores the same information as the operational database, deliberately arranged with repetition. Let us push the idea to its limit and build a third design: **one big table**, in which every order line carries every attribute of its date, customer and product.

In [ ]:
warehouse.execute("""
    CREATE TABLE sales_flat AS
    SELECT d.date, d.year, d.quarter, d.month_name, d.weekday, d.is_weekend,
           c.name AS customer, c.city, c.age_group,
           p.title, p.category, p.price_band,
           f.quantity, f.revenue
    FROM fact_sales AS f
    JOIN dim_date AS d     ON d.date_key = f.date_key
    JOIN dim_customer AS c ON c.customer_key = f.customer_key
    JOIN dim_product AS p  ON p.product_key = f.product_key
""")

flat_query = """
    SELECT category, year, ROUND(SUM(revenue)) AS revenue
    FROM sales_flat
    GROUP BY category, year
    ORDER BY category, year
"""
flat_rows, flat_time = timed(warehouse, flat_query)

print("The same question, three designs:\n")
print(f"  normalised (Notebook 01):  {operational_time:6.0f} ms   three tables joined, revenue computed")
print(f"  star schema:               {star_time:6.0f} ms   fact table joined with two dimensions")
print(f"  one big table:             {flat_time:6.0f} ms   no joins at all")

The more the data is arranged in advance for reading, the less work each question takes. That is the gain.

Now the cost. A customer moves from one city to another. How many rows must be changed in each design?

In [ ]:
customer = 1234

in_normalised = shop.execute("SELECT COUNT(*) FROM customers WHERE customer_id = ?", (customer,)).fetchone()[0]
in_star = warehouse.execute("SELECT COUNT(*) FROM dim_customer WHERE customer_key = ?", (customer,)).fetchone()[0]
in_flat = warehouse.execute("SELECT COUNT(*) FROM sales_flat WHERE customer = ?", (f"Customer {customer}",)).fetchone()[0]

def megabytes(db):
    return db.execute("PRAGMA page_count").fetchone()[0] * db.execute("PRAGMA page_size").fetchone()[0] / 1_000_000

flat_only = sqlite3.connect(":memory:")
flat_only.execute("CREATE TABLE sales_flat (date, year, quarter, month_name, weekday, is_weekend, customer, city, age_group, title, category, price_band, quantity, revenue)")
flat_only.executemany("INSERT INTO sales_flat VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)", warehouse.execute("SELECT * FROM sales_flat"))

star_only = sqlite3.connect(":memory:")
star_only.execute("CREATE TABLE fact_sales (date_key INTEGER, customer_key INTEGER, product_key INTEGER, quantity INTEGER, revenue REAL)")
star_only.executemany("INSERT INTO fact_sales VALUES (?,?,?,?,?)", fact_rows)

print(f"{'design':<16} {'rows to change when a customer moves':>37} {'size of the sales data':>24}")
print(f"{'normalised':<16} {in_normalised:>37} {'':>24}")
print(f"{'star schema':<16} {in_star:>37} {megabytes(star_only):>21.1f} MB")
print(f"{'one big table':<16} {in_flat:>37} {megabytes(flat_only):>21.1f} MB")

In the big table, the customer's city is written on every one of their order lines, and each must be found and changed, or the table will contradict itself. That is the **update anomaly** of Notebook 01, returning exactly as promised. The big table is also several times larger, because every description is repeated on every row.

| | Normalised | Star schema | One big table |
|---|---|---|---|
| Each fact stored | once | once per dimension row | on every row it applies to |
| Analytical queries | most joins, slowest | few joins | no joins, fastest |
| Changing a description | one row | one row | many rows |
| Storage | smallest | small | largest |
| Best for | running the business | analysis in general | a fixed set of reports, read constantly |

No design wins on every row of that table. The normalised design wins when the data changes all the time and questions are small. The flattened designs win when the data is loaded once and read thousands of times. The star schema is the widely used middle: nearly as convenient to query as one big table, and with each description still kept in a single place.

> ⚠️ **Common pitfalls**
>
> - Leaving the **grain** vague. If some rows of a fact table are order lines and others are whole orders, every total is wrong. Write the grain down in one sentence before creating the table.
> - Storing in the fact table something that cannot be **added up**, such as a unit price or a percentage, and then summing it. Store the quantities and amounts, and derive averages and ratios in the query.
> - Running analytical queries on the operational database. A report that reads every row competes with the tills for the same machine.
> - Recomputing the past from present values. Store what was true at the time of the event.

## 6. Keeping history

One problem remains, and it is specific to warehouses. A customer who lived in Volos moves to Athens. In the operational database we overwrite the city, as we should: the shop needs to know where to send the next parcel. If the warehouse does the same, all the customer's **past** purchases are suddenly attributed to Athens, and last year's sales by city change after the fact.

The warehouse's answer is not to overwrite. When a description changes, the old row is **closed** and a **new row** is added, each with the period during which it was valid. This technique is called a **slowly changing dimension** (of "type 2").

In [ ]:
history = sqlite3.connect(":memory:")
history.executescript("""
    CREATE TABLE dim_customer (customer_key INTEGER PRIMARY KEY, customer_id INTEGER, name TEXT, city TEXT,
                               valid_from TEXT, valid_to TEXT);
    CREATE TABLE fact_sales   (customer_key INTEGER, date TEXT, revenue REAL);

    INSERT INTO dim_customer VALUES (1, 77, 'Maria', 'Volos', '2023-01-01', '9999-12-31');
    INSERT INTO fact_sales VALUES (1, '2023-05-10', 40.0), (1, '2023-11-02', 25.0);
""")

def customer_moves(customer_id, new_city, on_date):
    name = history.execute("SELECT name FROM dim_customer WHERE customer_id = ? AND valid_to = '9999-12-31'", (customer_id,)).fetchone()[0]
    history.execute("UPDATE dim_customer SET valid_to = ? WHERE customer_id = ? AND valid_to = '9999-12-31'", (on_date, customer_id))
    history.execute("INSERT INTO dim_customer (customer_id, name, city, valid_from, valid_to) VALUES (?, ?, ?, ?, '9999-12-31')",
                    (customer_id, name, new_city, on_date))

customer_moves(77, "Athens", "2024-02-01")
new_key = history.execute("SELECT customer_key FROM dim_customer WHERE customer_id = 77 AND valid_to = '9999-12-31'").fetchone()[0]
history.execute("INSERT INTO fact_sales VALUES (?, '2024-06-20', 60.0)", (new_key,))

show(history, "SELECT * FROM dim_customer")
show(history, """
    SELECT c.city, SUM(f.revenue) AS revenue
    FROM fact_sales AS f JOIN dim_customer AS c ON c.customer_key = f.customer_key
    GROUP BY c.city
""")

Maria is now two rows in the dimension: one for her time in Volos, one for Athens. Her old purchases still point to the Volos row and her new one to the Athens row, so revenue by city is correct **for the time of each sale**. Note the two different keys: `customer_id` identifies the person, and `customer_key` identifies one version of her description.

## 7. Where we stand

| The problem | The answer |
|---|---|
| Analytical questions are awkward and slow on a normalised database | a separate copy arranged for reading: a warehouse |
| Every question is "a number by some descriptions" | a star schema: facts in the middle, dimensions around |
| Getting the data there | ETL: extract, transform, load |
| The past changes when a value is overwritten | store measures at the time of the event, and keep versions of dimensions |

This closes Module A. In four notebooks the data stopped fitting in one table, then a query stopped fitting in the time we had, and then the questions stopped fitting the design. Through all of it one assumption held: **data is rows in tables**. The next module examines that assumption from two sides. Notebook 05 looks underneath the table, at how the bytes are actually laid out in a file, and finds that the layout matters a great deal for analysis. Notebook 06 meets data that does not want to be a table at all.

---
## ✏️ Exercises

### Exercise 1 (Read the star)
Using the warehouse, find the three **cities** with the highest revenue in the last quarter of 2024 (`2024-Q4`).

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show(warehouse, """
    SELECT c.city, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f
    JOIN dim_customer AS c ON c.customer_key = f.customer_key
    JOIN dim_date AS d     ON d.date_key = f.date_key
    WHERE d.quarter = '2024-Q4'
    GROUP BY c.city
    ORDER BY revenue DESC
    LIMIT 3
""")
```

*Fact table in the middle, the two dimensions the question mentions, a filter on one of them, a group on the other. Almost every warehouse query is a variation on this.*
</details>

### Exercise 2 (Roll up and drill down)
Show revenue by **price band**. Then drill down: revenue by price band **and** age group. Which combination brings in the most money?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show(warehouse, """
    SELECT p.price_band, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f JOIN dim_product AS p ON p.product_key = f.product_key
    GROUP BY p.price_band ORDER BY revenue DESC
""")
show(warehouse, """
    SELECT p.price_band, c.age_group, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f
    JOIN dim_product AS p  ON p.product_key = f.product_key
    JOIN dim_customer AS c ON c.customer_key = f.customer_key
    GROUP BY p.price_band, c.age_group
    ORDER BY revenue DESC
""", limit=9)
```

*Drilling down only added one dimension to the join and one column to the grouping. The expensive books bring in the most revenue, and the largest age group buys the most, simply because there are more people in it. Before reading meaning into a breakdown, check how many customers are in each group.*
</details>

### Exercise 3 (Extend a dimension)
Analysts want to compare the **seasons**. Add a column `season` to `dim_date` (December to February is winter, March to May spring, June to August summer, September to November autumn) and fill it in. Then show revenue by season and year.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
warehouse.execute("ALTER TABLE dim_date ADD COLUMN season TEXT")
warehouse.execute("""
    UPDATE dim_date SET season = CASE
        WHEN month IN (12, 1, 2) THEN 'winter'
        WHEN month IN (3, 4, 5)  THEN 'spring'
        WHEN month IN (6, 7, 8)  THEN 'summer'
        ELSE 'autumn' END
""")
warehouse.commit()

show(warehouse, """
    SELECT d.year, d.season, ROUND(SUM(f.revenue)) AS revenue
    FROM fact_sales AS f JOIN dim_date AS d ON d.date_key = f.date_key
    GROUP BY d.year, d.season
    ORDER BY d.year, revenue DESC
""")
```

*We changed 731 rows of a small table, and a new way of looking at every sale became available. The hundreds of thousands of fact rows were not touched. This is the advantage of the star over one big table, where the new column would have to be written on every row.*
</details>

### Exercise 4 (A summary table)
Reports that are requested every day can be computed once and stored. Create a table `monthly_sales` holding revenue and copies sold for each year, month and category. Time a query for the revenue of `Travel` books in each month of 2024, once against `fact_sales` with its dimensions and once against `monthly_sales`. What is the risk of keeping such a table?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
warehouse.execute("""
    CREATE TABLE monthly_sales AS
    SELECT d.year, d.month, p.category, SUM(f.revenue) AS revenue, SUM(f.quantity) AS copies
    FROM fact_sales AS f
    JOIN dim_date AS d    ON d.date_key = f.date_key
    JOIN dim_product AS p ON p.product_key = f.product_key
    GROUP BY d.year, d.month, p.category
""")

from_facts = """
    SELECT d.month, ROUND(SUM(f.revenue)) FROM fact_sales AS f
    JOIN dim_date AS d ON d.date_key = f.date_key JOIN dim_product AS p ON p.product_key = f.product_key
    WHERE d.year = 2024 AND p.category = 'Travel' GROUP BY d.month ORDER BY d.month
"""
from_summary = "SELECT month, ROUND(revenue) FROM monthly_sales WHERE year = 2024 AND category = 'Travel' ORDER BY month"

rows_a, time_a = timed(warehouse, from_facts)
rows_b, time_b = timed(warehouse, from_summary)
print(f"From the fact table:    {time_a:8.2f} ms")
print(f"From the summary table: {time_b:8.2f} ms   ({warehouse.execute('SELECT COUNT(*) FROM monthly_sales').fetchone()[0]} rows)")
print("Same answer:", rows_a == rows_b)
```

*The summary table has a couple of hundred rows where the fact table has a quarter of a million, so the query is far faster. The risk is that it goes stale: when new sales are loaded, the summary must be rebuilt or it will disagree with the facts. A table like this, stored and refreshed by the database itself, is called a materialised view.*
</details>

### Exercise 5 (History, a little harder)
In the `history` database of Section 6, Maria moves again, to Patras, on 2024-09-01, and then buys books worth 35.0 on 2024-10-05. Record both events using `customer_moves`. Then answer two questions with SQL: (a) what is her revenue in each city, and (b) where did she live on 2024-03-15?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
customer_moves(77, "Patras", "2024-09-01")
current_key = history.execute("SELECT customer_key FROM dim_customer WHERE customer_id = 77 AND valid_to = '9999-12-31'").fetchone()[0]
history.execute("INSERT INTO fact_sales VALUES (?, '2024-10-05', 35.0)", (current_key,))

show(history, """
    SELECT c.city, SUM(f.revenue) AS revenue
    FROM fact_sales AS f JOIN dim_customer AS c ON c.customer_key = f.customer_key
    GROUP BY c.city
""")
show(history, """
    SELECT name, city FROM dim_customer
    WHERE customer_id = 77 AND valid_from <= '2024-03-15' AND valid_to > '2024-03-15'
""")
```

*Three versions of one customer, each sale attached to the version that was current when it happened. The validity dates also let us ask what was true on any past day, which an operational database, having overwritten the old values, could never answer.*
</details>

## 🔑 Key takeaways

- **OLTP** systems run a business with many small reads and writes. **OLAP** systems analyse it with large read-only summaries. They need different designs.
- A **data warehouse** is a separate copy of the data, arranged for analysis and filled by **ETL**.
- A **star schema** has a **fact table** (one row per event, at a stated **grain**, with additive **measures**) surrounded by denormalised **dimension tables**.
- Analysts **roll up**, **drill down** and **slice**: all are a join to a dimension plus a `GROUP BY`.
- Denormalising makes reading faster and simpler, and makes changing data and storage more expensive. Which design is best depends on whether the data is mostly **written** or mostly **read**.
- A warehouse must preserve **history**: store measures as they were, and keep old versions of dimension rows.

### 🏁 You have completed Module A: Data at Rest!
You can design tables, query and join them, make queries fast and say why they are fast, and arrange data for analysis. Everything so far has assumed that data is rows in tables, held by one database on one machine.

**Next:** *Module B: Beyond Tables*, starting with *Notebook 05: Files and Formats*, where we look at how a table is laid out on disk, and discover that turning it on its side makes analysis many times faster.

---
*End of Module A.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*